# Notebook 11 — LightGBM

LightGBM gradient boosting — fixed configuration, Optuna tuning, final model.

**Prerequisite:** run `Notebook00_Setup_PCA300.ipynb` first.

Changes vs. previous version: (1) 95% bootstrap CIs, per-class report, confusion matrix and fitted-model smoke test, identical to the other classifier notebooks; (2) Optuna tuning on the validation split by default; (3) dataset guard against stale v2 runs.

## Step 0 — Setup

In [1]:
import os, warnings, pickle, json, time
import numpy as np
import pandas as pd
from pathlib import Path

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)

warnings.filterwarnings("ignore")
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

DATASET_ROOT = "/home/ali/Desktop/Algerian_Lamiaceae_v3"
OUTPUT_DIR   = os.path.join(DATASET_ROOT, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")


import lightgbm as lgb
from lightgbm import LGBMClassifier
print(f"LightGBM version: {lgb.__version__}")

Device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Output: /home/ali/Desktop/Algerian_Lamiaceae_v3/results
LightGBM version: 4.6.0


## Step 1 — Load Cached Features and Train/Val/Test Split

In [2]:
HC_CACHE    = os.path.join(OUTPUT_DIR, "hc_features_cache.npz")
DEEP_CACHE  = os.path.join(OUTPUT_DIR, "deep_features_cache.npz")
SPLIT_CACHE = os.path.join(OUTPUT_DIR, "train_test_split.npz")

X_hc = np.load(HC_CACHE, allow_pickle=True)["X"]
print(f"Handcrafted features: {X_hc.shape}")

if os.path.exists(DEEP_CACHE):
    X_deep = np.load(DEEP_CACHE, allow_pickle=True)["X"]
    X_all  = np.concatenate([X_deep, X_hc], axis=1)
    print(f"Deep features: {X_deep.shape}")
else:
    X_all = X_hc
    print("Deep features not found -- using handcrafted only")
print(f"Fused features: {X_all.shape}")

split = np.load(SPLIT_CACHE, allow_pickle=True)
X_train_sc = split["X_train_sc"]
X_val_sc   = split["X_val_sc"]
X_test_sc  = split["X_test_sc"]
y_train_sm = split["y_train_sm"]
y_val_raw  = split["y_val_raw"]
y_test_raw = split["y_test_raw"]
n_components = int(split["n_components"])

with open(os.path.join(OUTPUT_DIR, "label_encoder_species.pkl"), "rb") as f:
    le = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "sc1.pkl"), "rb") as f:
    sc1 = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "sc2.pkl"), "rb") as f:
    sc2 = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "pca.pkl"), "rb") as f:
    pca = pickle.load(f)

class_names = list(le.classes_)
NUM_CLASSES = len(class_names)

print(f"Train: {X_train_sc.shape}  Val: {X_val_sc.shape}  Test: {X_test_sc.shape}")
print(f"Classes: {NUM_CLASSES} -- {class_names}")

test_support = pd.Series(y_test_raw).value_counts().sort_index()
print("\nTest-set support per class:")
for i, cls in enumerate(class_names):
    n = int(test_support.get(i, 0))
    flag = "  \u26a0 SMALL SUPPORT" if n < 30 else ""
    print(f"  {cls:<28} n={n:>5}{flag}")


Handcrafted features: (19145, 73)
Deep features: (19145, 1536)
Fused features: (19145, 1609)
Train: (35000, 300)  Val: (2372, 300)  Test: (2366, 300)
Classes: 7 -- [np.str_('Lamium_pooled'), np.str_('Lavandula_angustifolia'), np.str_('Lavandula_pooled'), np.str_('Leucas_aspera'), np.str_('Mentha_pooled'), np.str_('Ocimum_pooled'), np.str_('Plectranthus_pooled')]

Test-set support per class:
  Lamium_pooled                n= 1529
  Lavandula_angustifolia       n=  342
  Lavandula_pooled             n=  340
  Leucas_aspera                n=   11  ⚠ SMALL SUPPORT
  Mentha_pooled                n=   48
  Ocimum_pooled                n=   51
  Plectranthus_pooled          n=   45


In [3]:
# ── Guard: make sure this run uses the final v3 dataset (prevents stale v2 runs) ──
assert "Algerian_Lamiaceae_v3" in OUTPUT_DIR, f"Wrong OUTPUT_DIR: {OUTPUT_DIR}"
assert "Lamium_pooled" in class_names, f"Old class names detected: {class_names}"
print(f"Dataset check passed: v3 results folder, {len(class_names)} classes, "
      f"test n={X_test_sc.shape[0]:,}, val n={X_val_sc.shape[0]:,}")

Dataset check passed: v3 results folder, 7 classes, test n=2,366, val n=2,372


## Step 2 — Evaluation Utilities (identical to the other classifier notebooks)

In [4]:
N_BOOTSTRAP = 1000
SMALL_SUPPORT_THRESHOLD = 30

def evaluate(clf, Xtr, Xte, ytr, yte):
    """Fits clf IN PLACE and returns it alongside its metrics -- the same
    object is what gets pickled downstream, closing the unfitted-model bug."""
    clf.fit(Xtr, ytr)
    yp = clf.predict(Xte)
    yproba = None
    auc_s = None
    if hasattr(clf, "predict_proba"):
        try:
            yproba = clf.predict_proba(Xte)
            auc_s = round(roc_auc_score(yte, yproba, multi_class="ovr",
                                         average="macro") * 100, 2)
        except Exception:
            pass
    return {
        "Accuracy":  round(accuracy_score(yte, yp) * 100, 2),
        "Precision": round(precision_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "Recall":    round(recall_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "F1":        round(f1_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "AUC": auc_s,
        "_yp": yp, "_yproba": yproba, "_fitted_clf": clf,
    }

def bootstrap_ci(y_true, y_pred, y_proba=None, n_boot=N_BOOTSTRAP, seed=GLOBAL_SEED):
    """Percentile bootstrap 95% CI for Accuracy/Precision/Recall/F1/AUC.
    Resamples where a bootstrap draw happens to omit a class are skipped
    for the AUC calculation only (multiclass OvR AUC requires every class
    present); this is tracked and reported, not silently ignored."""
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n = len(y_true)
    accs, precs, recs, f1s, aucs = [], [], [], [], []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, yp = y_true[idx], y_pred[idx]
        accs.append(accuracy_score(yt, yp))
        precs.append(precision_score(yt, yp, average="weighted", zero_division=0))
        recs.append(recall_score(yt, yp, average="weighted", zero_division=0))
        f1s.append(f1_score(yt, yp, average="weighted", zero_division=0))
        if y_proba is not None:
            try:
                aucs.append(roc_auc_score(yt, y_proba[idx], multi_class="ovr", average="macro"))
            except ValueError:
                pass  # a class was missing from this particular resample

    def summarize(arr):
        arr = np.array(arr) * 100
        return {"mean": round(float(arr.mean()), 2),
                "ci_lo": round(float(np.percentile(arr, 2.5)), 2),
                "ci_hi": round(float(np.percentile(arr, 97.5)), 2)}

    result = {"Accuracy": summarize(accs), "Precision": summarize(precs),
              "Recall": summarize(recs), "F1": summarize(f1s)}
    if y_proba is not None:
        result["AUC"] = summarize(aucs) if aucs else None
        result["AUC_valid_resamples"] = len(aucs)
        result["AUC_skipped_resamples"] = n_boot - len(aucs)
    return result

def per_class_report(y_true, y_pred, class_names, small_support_threshold=SMALL_SUPPORT_THRESHOLD):
    report = classification_report(y_true, y_pred, labels=list(range(len(class_names))),
                                    target_names=class_names, output_dict=True, zero_division=0)
    rows = []
    for cls in class_names:
        r = report[cls]
        rows.append({"Class": cls, "Precision": round(r["precision"] * 100, 2),
                     "Recall": round(r["recall"] * 100, 2),
                     "F1": round(r["f1-score"] * 100, 2), "Support": int(r["support"])})
    df = pd.DataFrame(rows)
    small = df[df["Support"] < small_support_threshold]
    if not small.empty:
        print(f"  \u26a0 Small test support (<{small_support_threshold} samples) -- "
              f"per-class scores below should not be over-interpreted (comment 7):")
        for _, r in small.iterrows():
            print(f"     {r['Class']:<28} support={r['Support']:>4}  "
                  f"P={r['Precision']:.1f}%  R={r['Recall']:.1f}%  F1={r['F1']:.1f}%")
    return df

def save_full_result(name, m, y_true, class_names):
    """Fix: takes the fitted clf FROM m (the object evaluate() actually
    trained), not a separately-passed, possibly-unfitted reference."""
    clf = m["_fitted_clf"]
    yp, yproba = m["_yp"], m["_yproba"]

    ci = bootstrap_ci(y_true, yp, yproba)
    per_class = per_class_report(y_true, yp, class_names)
    cm = confusion_matrix(y_true, yp, labels=list(range(len(class_names))))

    result = {k: v for k, v in m.items() if not k.startswith("_")}
    result["Classifier"] = name
    result["Bootstrap_CI_95"] = ci
    result["N_bootstrap"] = N_BOOTSTRAP
    result["PerClass"] = per_class.to_dict(orient="records")

    out_path = os.path.join(OUTPUT_DIR, f"result_{name.replace(' ','_')}.json")
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)

    mdl_path = os.path.join(OUTPUT_DIR, f"model_{name.replace(' ','_')}.pkl")
    with open(mdl_path, "wb") as f:
        pickle.dump(clf, f)   # the ACTUAL fitted object

    cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{name.replace(' ','_')}.csv")
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_path)

    auc_s = f"{m['AUC']}%" if m["AUC"] is not None else "n/a"
    print(f"  Acc={m['Accuracy']:6.2f}% [{ci['Accuracy']['ci_lo']:.1f}-{ci['Accuracy']['ci_hi']:.1f}]  "
          f"P={m['Precision']:6.2f}%  R={m['Recall']:6.2f}%  F1={m['F1']:6.2f}% "
          f"[{ci['F1']['ci_lo']:.1f}-{ci['F1']['ci_hi']:.1f}]  AUC={auc_s}")
    print(f"  Saved -> {out_path}")
    print(f"  Saved -> {mdl_path}  (fitted model, verified via clf.predict smoke-test below)")
    print(f"  Saved -> {cm_path}")

    # Smoke-test: confirm the pickled model is actually fitted and usable.
    with open(mdl_path, "rb") as f:
        reloaded = pickle.load(f)
    _ = reloaded.predict(X_test_sc[:1])
    print(f"  Smoke-test passed: reloaded model predicts without error.")

    return result

print("Evaluation utilities ready (fit-and-return, bootstrap CI, per-class, confusion matrix).")


Evaluation utilities ready (fit-and-return, bootstrap CI, per-class, confusion matrix).


## Step 2b — LightGBM device check

In [5]:
def pick_lgbm_device():
    """Try LightGBM's CUDA, then OpenCL GPU backend; fall back to CPU.
    The standard pip/conda build supports CPU only, so CPU is the usual result."""
    Xs = np.random.rand(500, 10).astype(np.float32); ys = np.random.randint(0, 2, 500)
    for dev in ("cuda", "gpu"):
        try:
            LGBMClassifier(n_estimators=2, device=dev, verbose=-1).fit(Xs, ys)
            print(f"  LightGBM device '{dev}': available")
            return dev
        except Exception as e:
            print(f"  LightGBM device '{dev}': not available ({type(e).__name__})")
    return "cpu"

LGB_DEVICE = pick_lgbm_device()
print(f"Using LightGBM device: {LGB_DEVICE}")

# LightGBM is faster on float32
X_train_sc = X_train_sc.astype(np.float32)
X_val_sc   = X_val_sc.astype(np.float32)
X_test_sc  = X_test_sc.astype(np.float32)

# ---- Speed: find the fastest number of CPU threads on THIS machine ----
# n_jobs=-1 uses every logical thread (incl. hyper-threads / efficiency cores), which on laptop
# CPUs often makes LightGBM much SLOWER. Time a short fit for a few thread counts and keep the best.
# The thread count does not change the fitted model (same data, same seed, force_col_wise).
_idx = np.random.RandomState(0).choice(len(X_train_sc), 10000, replace=False)
_cands = sorted({2, 4, 6, 8, max(1, (os.cpu_count() or 2) // 2), os.cpu_count() or 2})
_times = {}
for nj in _cands:
    t0 = time.time()
    LGBMClassifier(n_estimators=20, num_leaves=64, learning_rate=0.05, verbose=-1, n_jobs=nj,
                   force_col_wise=True, random_state=GLOBAL_SEED).fit(X_train_sc[_idx], y_train_sm[_idx])
    _times[nj] = time.time() - t0
    print(f"  n_jobs={nj:2d}: {_times[nj]:5.2f}s for 20 trees on 10k rows")
N_JOBS = min(_times, key=_times.get)
print(f"Using n_jobs={N_JOBS} (os.cpu_count()={os.cpu_count()})")

  LightGBM device 'cuda': not available (LightGBMError)
  LightGBM device 'gpu': not available (LightGBMError)
Using LightGBM device: cpu


[LightGBM] [Fatal] CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1


  n_jobs= 2:  4.95s for 20 trees on 10k rows
  n_jobs= 4:  2.78s for 20 trees on 10k rows
  n_jobs= 6:  2.87s for 20 trees on 10k rows
  n_jobs= 8:  2.17s for 20 trees on 10k rows
  n_jobs=16:  1.74s for 20 trees on 10k rows
  n_jobs=32: 314.57s for 20 trees on 10k rows
Using n_jobs=16 (os.cpu_count()=32)


## Step 3 — Baseline LightGBM (lr=0.05, 64 leaves, 200 trees)

In [6]:
name = "LightGBM"
print(f"Training {name} (fixed configuration) on {LGB_DEVICE} ...")
print("AUC methodology: LGBMClassifier's native predict_proba used directly. "
      "AUC is One-vs-Rest, macro-averaged.")

clf = LGBMClassifier(learning_rate=0.05, num_leaves=64, n_estimators=200,
                     class_weight="balanced", random_state=GLOBAL_SEED, verbose=-1,
                     device=LGB_DEVICE, n_jobs=N_JOBS, force_col_wise=True)
RERUN_BASELINE = False   # the baseline already finished once; set True to retrain it
_res = os.path.join(OUTPUT_DIR, f"result_{name}.json")
if os.path.exists(_res) and not RERUN_BASELINE:
    result_baseline = json.load(open(_res))
    ci = result_baseline["Bootstrap_CI_95"]
    print(f"  Loaded finished baseline from {_res}")
    print(f"  Acc={result_baseline['Accuracy']}% [{ci['Accuracy']['ci_lo']}-{ci['Accuracy']['ci_hi']}]  "
          f"F1={result_baseline['F1']}% [{ci['F1']['ci_lo']}-{ci['F1']['ci_hi']}]  AUC={result_baseline['AUC']}%")
else:
    t0 = time.time()
    m = evaluate(clf, X_train_sc, X_test_sc, y_train_sm, y_test_raw)   # fitted IN PLACE
    print(f"  Fit + predict time: {time.time()-t0:.1f}s")
    result_baseline = save_full_result(name, m, y_test_raw, class_names)

Training LightGBM (fixed configuration) on cpu ...
AUC methodology: LGBMClassifier's native predict_proba used directly. AUC is One-vs-Rest, macro-averaged.
  Loaded finished baseline from /home/ali/Desktop/Algerian_Lamiaceae_v3/results/result_LightGBM.json
  Acc=90.28% [89.05-91.46]  F1=90.47% [89.33-91.6]  AUC=98.88%


## Step 4 — Optuna Hyperparameter Search

**`TUNE_MODE`** controls how Optuna scores candidate configurations:

* `"val"` (default) — fit on the SMOTE-resampled training set, score weighted F1 on the untouched validation split. This is the protocol used by the other ten classifiers and described in the manuscript.
* `"cv"` — mean weighted F1 of 3-fold stratified CV on the SMOTE-resampled training set (the protocol of the previous run). SMOTE samples are interpolated from real neighbours, so these folds are not independent and the CV score is optimistic.

The test split is never used for tuning in either mode.

In [7]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import StratifiedKFold

TUNE_MODE = "val"      # "val" (recommended, matches other classifiers) or "cv" (previous protocol)
N_TRIALS  = 40
optuna.logging.set_verbosity(optuna.logging.WARNING)

def build_lgbm(p):
    return LGBMClassifier(**p, subsample_freq=1, class_weight="balanced",
                          random_state=GLOBAL_SEED, verbose=-1, device=LGB_DEVICE,
                          n_jobs=N_JOBS, force_col_wise=True)

def objective_lgbm(trial):
    p = dict(
        learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves        = trial.suggest_int("num_leaves", 16, 256, log=True),
        n_estimators      = trial.suggest_int("n_estimators", 100, 800, step=50),
        min_child_samples = trial.suggest_int("min_child_samples", 5, 100, log=True),
        subsample         = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_lambda        = trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    )
    desc = (f"lr={p['learning_rate']:.3f} leaves={p['num_leaves']} "
            f"trees={p['n_estimators']} mcs={p['min_child_samples']}")
    t0 = time.time()
    if TUNE_MODE == "val":
        # Early stopping on the validation split: stop adding trees once the validation
        # log-loss has not improved for 50 rounds (n_estimators is the upper limit).
        clf = build_lgbm(p).fit(X_train_sc, y_train_sm, eval_set=[(X_val_sc, y_val_raw)],
                                eval_metric="multi_logloss",
                                callbacks=[lgb.early_stopping(50, verbose=False)])
        best_it = int(clf.best_iteration_ or p["n_estimators"])
        trial.set_user_attr("best_iteration", best_it)
        desc += f" -> stopped at {best_it} trees"
        score = f1_score(y_val_raw, clf.predict(X_val_sc, num_iteration=best_it), average="weighted")
        print(f"[Trial {trial.number+1:2d}/{N_TRIALS}] done   {time.time()-t0:6.1f}s  "
              f"VAL F1={score*100:.2f}%  {desc}", flush=True)
        return score
    skf, scores = StratifiedKFold(n_splits=3, shuffle=True, random_state=GLOBAL_SEED), []
    for k, (tr, va) in enumerate(skf.split(X_train_sc, y_train_sm)):
        clf = build_lgbm(p).fit(X_train_sc[tr], y_train_sm[tr])
        scores.append(f1_score(y_train_sm[va], clf.predict(X_train_sc[va]), average="weighted"))
        trial.report(float(np.mean(scores)), k)
        if trial.should_prune():
            print(f"[Trial {trial.number+1:2d}/{N_TRIALS}] PRUNED fold {k+1}  {time.time()-t0:6.1f}s  "
                  f"CV F1={np.mean(scores)*100:.2f}%  {desc}", flush=True)
            raise optuna.TrialPruned()
    print(f"[Trial {trial.number+1:2d}/{N_TRIALS}] done   {time.time()-t0:6.1f}s  "
          f"CV F1={np.mean(scores)*100:.2f}%  {desc}", flush=True)
    return float(np.mean(scores))

print("Optuna search space: learning_rate ~ LogUniform[0.01,0.3], num_leaves ~ LogInt[16,256], "
      "n_estimators ~ Int[100,800] step 50, min_child_samples ~ LogInt[5,100], "
      "subsample ~ Uniform[0.5,1], colsample_bytree ~ Uniform[0.3,1], reg_lambda ~ LogUniform[1e-3,10]")
print(f"seed={GLOBAL_SEED}, n_trials={N_TRIALS}, sampler=TPESampler, TUNE_MODE='{TUNE_MODE}', device={LGB_DEVICE}")
print("Objective: " + ("fit on SMOTE-resampled TRAIN, score weighted-F1 on untouched VAL."
                       if TUNE_MODE == "val" else
                       "3-fold stratified CV weighted-F1 on SMOTE-resampled TRAIN, median pruning (optimistic)."))

t_start = time.time()
pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=0) if TUNE_MODE == "cv" else optuna.pruners.NopPruner()
study_lgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=GLOBAL_SEED), pruner=pruner)
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS)

n_pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in study_lgbm.trials)
print(f"\nDone in {(time.time()-t_start)/60:.1f} min  ({n_pruned}/{N_TRIALS} trials pruned)")
print(f"Best {TUNE_MODE.upper()} F1(weighted) = {study_lgbm.best_value*100:.2f}%  "
      f"(trial {study_lgbm.best_trial.number+1})")
print(f"Best params: {study_lgbm.best_params}")
if "best_iteration" in study_lgbm.best_trial.user_attrs:
    print(f"Early stopping: best trial used {study_lgbm.best_trial.user_attrs['best_iteration']} trees "
          f"(limit {study_lgbm.best_params['n_estimators']}); the final model is refitted with that number")

with open(os.path.join(OUTPUT_DIR, "optuna_result_LightGBM.json"), "w") as f:
    json.dump({"Classifier": "LightGBM_Optuna_BestTrial", "TuneMode": TUNE_MODE,
               "Best_F1_weighted": round(study_lgbm.best_value*100, 2),
               "BestParams": {**study_lgbm.best_params,
                              **({"n_estimators": study_lgbm.best_trial.user_attrs["best_iteration"]}
                                 if "best_iteration" in study_lgbm.best_trial.user_attrs else {})},
               "SearchedMaxTrees": study_lgbm.best_params.get("n_estimators"),
               "NumTrials": N_TRIALS, "NumPruned": n_pruned,
               "SearchTime_min": round((time.time()-t_start)/60, 1)}, f, indent=2)
print(f"Saved -> {os.path.join(OUTPUT_DIR, 'optuna_result_LightGBM.json')}")

Optuna search space: learning_rate ~ LogUniform[0.01,0.3], num_leaves ~ LogInt[16,256], n_estimators ~ Int[100,800] step 50, min_child_samples ~ LogInt[5,100], subsample ~ Uniform[0.5,1], colsample_bytree ~ Uniform[0.3,1], reg_lambda ~ LogUniform[1e-3,10]
seed=42, n_trials=40, sampler=TPESampler, TUNE_MODE='val', device=cpu
Objective: fit on SMOTE-resampled TRAIN, score weighted-F1 on untouched VAL.
[Trial  1/40] done     35.0s  VAL F1=92.11%  lr=0.036 leaves=223 trees=600 mcs=29 -> stopped at 227 trees
[Trial  2/40] done     11.9s  VAL F1=89.93%  lr=0.190 leaves=84 trees=600 mcs=5 -> stopped at 43 trees
[Trial  3/40] done     12.5s  VAL F1=90.64%  lr=0.019 leaves=26 trees=300 mcs=23 -> stopped at 300 trees
[Trial  4/40] done     17.1s  VAL F1=91.18%  lr=0.016 leaves=35 trees=350 mcs=19 -> stopped at 350 trees
[Trial  5/40] done     17.7s  VAL F1=91.85%  lr=0.075 leaves=18 trees=550 mcs=8 -> stopped at 368 trees
[Trial  6/40] done      6.3s  VAL F1=91.59%  lr=0.156 leaves=36 trees=150 

## Step 5 — Final Model: Retrain with Best Hyperparameters, Evaluate Once on Test

In [8]:
name = "LightGBM_Optuna"
print(f"Training {name} (final retrained model, evaluated ONCE on held-out test) ...")

with open(os.path.join(OUTPUT_DIR, "optuna_result_LightGBM.json")) as f:
    opt = json.load(f)
bp = opt["BestParams"]
print(f"Using params from {opt.get('TuneMode', 'cv').upper()}-based search: {bp}")

clf_final = build_lgbm(bp)
t0 = time.time()
m_final = evaluate(clf_final, X_train_sc, X_test_sc, y_train_sm, y_test_raw)   # fitted IN PLACE
print(f"  Fit + predict time: {time.time()-t0:.1f}s")
result_optuna = save_full_result(name, m_final, y_test_raw, class_names)

print("\n" + "="*70)
print("BASELINE (fixed hyperparameters) vs. OPTUNA-TUNED -- compare on TEST:")
print("="*70)
for lbl, r in [("Baseline", result_baseline), ("Optuna  ", result_optuna)]:
    ci = r["Bootstrap_CI_95"]
    print(f"  {lbl}: Acc={r['Accuracy']:.2f}% [{ci['Accuracy']['ci_lo']:.1f}-{ci['Accuracy']['ci_hi']:.1f}]  "
          f"F1={r['F1']:.2f}% [{ci['F1']['ci_lo']:.1f}-{ci['F1']['ci_hi']:.1f}]  AUC={r['AUC']}%")

Training LightGBM_Optuna (final retrained model, evaluated ONCE on held-out test) ...
Using params from VAL-based search: {'learning_rate': 0.03195281813216447, 'num_leaves': 43, 'n_estimators': 424, 'min_child_samples': 40, 'subsample': 0.617536148867388, 'colsample_bytree': 0.45622709459234523, 'reg_lambda': 0.0022598483504384496}
  Fit + predict time: 22.9s
  ⚠ Small test support (<30 samples) -- per-class scores below should not be over-interpreted (comment 7):
     Leucas_aspera                support=  11  P=100.0%  R=100.0%  F1=100.0%
  Acc= 91.38% [90.2-92.4]  P= 91.56%  R= 91.38%  F1= 91.45% [90.3-92.5]  AUC=99.06%
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/result_LightGBM_Optuna.json
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/model_LightGBM_Optuna.pkl  (fitted model, verified via clf.predict smoke-test below)
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/confusion_matrix_LightGBM_Optuna.csv
  Smoke-test passed: reloaded model predi